# DISQCO-MFPQC Benchmark Circuits

The circuits generated by the notebook might not exactly match the original ones due to differences in random seeds, transpilation, or other factors. However, the circuits are generated using the same parameters and methods as the original ones, so they should be very similar in structure and characteristics.

In [ ]:
import random
import numpy as np


# Seed for reproducibility
SEED = 42
random.seed(SEED)

# Numpy seed for reproducibility in QAOA generation
np.random.seed(SEED)

In [ ]:
import re
from pathlib import Path

from qiskit import QuantumCircuit
from qiskit import qasm2
from qiskit import transpile
from qiskit.circuit.library import QFT, QuantumVolume
from tqdm import tqdm

from disqco.circuits.QAOA import QAOA_random
from disqco.circuits.cp_fraction import cp_fraction

In [ ]:
OUTPUT_DIR = "../../benchmarks"
QASMBENCH_PATH = "../../benchmarks/qasmbench"

In [ ]:
# Used in QFT, QV, and QAOA circuits
STANDARD_SIZES = [16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96]

BASIS_GATES = ["u", "cp"]

NUM_ITERATIONS = 10

In [ ]:
def write_circuit_pair(
    target_dir: Path,
    base_name: str,
    original_circuit,
    transpiled_circuit,
) -> None:
    target_dir.mkdir(parents=True, exist_ok=True)
    with (target_dir / f"{base_name}.qasm").open("w") as handle:
        handle.write(qasm2.dumps(original_circuit))
    with (target_dir / f"{base_name}.transpiled.qasm").open("w") as handle:
        handle.write(qasm2.dumps(transpiled_circuit))


def resolve_qasmbench_path(qasmbench_path: str | Path) -> Path:
    path = Path(qasmbench_path)
    if path.is_absolute():
        return path
    return (Path.cwd() / path).resolve()


def load_qasmbench_circuits_from_files(
    qasmbench_root: str | Path,
    category: str,
    min_qubits: int,
    max_qubits: int,
    max_depth: int,
    basis_gates: list[str],
    skip_patterns: list[str] | set[str] | tuple[str, ...] | None = None,
) -> list[tuple[str, object, object]]:
    root = Path(qasmbench_root)
    category_dir = root / category
    if not category_dir.exists():
        raise FileNotFoundError(f"QASMBench category folder not found: {category_dir}")

    patterns = list(skip_patterns or [])
    circuits: list[tuple[str, object, object]] = []

    for qasm_file in sorted(category_dir.rglob("*.qasm")):
        circuit_name = qasm_file.stem
        if any(re.match(pattern, circuit_name) for pattern in patterns):
            continue

        try:
            circuit = QuantumCircuit.from_qasm_file(str(qasm_file))
        except Exception:
            continue

        if not (min_qubits <= circuit.num_qubits < max_qubits):
            continue
        if circuit.depth() >= max_depth:
            continue

        transpiled_circuit = transpile(circuit, basis_gates=basis_gates)
        circuits.append((circuit_name, circuit, transpiled_circuit))

    return circuits


Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## QAOA

In [ ]:
QAOA_SIZES = STANDARD_SIZES
QAOA_PROB = 0.5
QAOA_REPS = 1

prob_tag = f"prob{str(QAOA_PROB).replace('.', 'p')}"
# reps_tag = f"reps{QAOA_REPS}"
for num_qubits in tqdm(QAOA_SIZES, desc="QAOA sizes"):
    for iteration_id in range(1, NUM_ITERATIONS + 1):
        current_seed = SEED + iteration_id
        random.seed(current_seed)
        np.random.seed(current_seed)

        original_circuit = QAOA_random(
            num_qubits,
            prob=QAOA_PROB,
            reps=QAOA_REPS,
        )
        transpiled_circuit = transpile(
            original_circuit,
            basis_gates=BASIS_GATES,
            seed_transpiler=current_seed,
            optimization_level=3,
        )

        target_dir = (
            Path(OUTPUT_DIR) / "disqco" / "mfpqc" / "qaoa" / f"n{num_qubits}".lower()
            # / prob_tag.lower()
            # / reps_tag.lower()
        )
        file_base = f"qaoa_n{num_qubits}_{prob_tag}_{iteration_id}"
        # file_base = f"qaoa_q{num_qubits}_{prob_tag}_{reps_tag}_{iteration_id}"
        write_circuit_pair(target_dir, file_base, original_circuit, transpiled_circuit)

## QFT

In [ ]:
QFT_SIZES = STANDARD_SIZES
QFT_DO_SWAPS = False

for num_qubits in tqdm(QFT_SIZES, desc="QFT sizes"):
    original_circuit = QFT(
        num_qubits,
        do_swaps=QFT_DO_SWAPS,
    )
    transpiled_circuit = transpile(
        original_circuit,
        basis_gates=BASIS_GATES,
        seed_transpiler=current_seed,
    )

    target_dir = (
        Path(OUTPUT_DIR) / "disqco" / "mfpqc" / "qft" / f"n{num_qubits}".lower()
    )
    file_base = f"qft_n{num_qubits}"
    write_circuit_pair(target_dir, file_base, original_circuit, transpiled_circuit)

## QV

In [ ]:
QV_SIZES = STANDARD_SIZES

for num_qubits in tqdm(QV_SIZES, desc="QV sizes"):
    for iteration_id in range(1, NUM_ITERATIONS + 1):
        current_seed = SEED + iteration_id
        random.seed(current_seed)
        np.random.seed(current_seed)

        original_circuit = QuantumVolume(
            num_qubits,
            num_qubits,
            seed=current_seed,
        )
        transpiled_circuit = transpile(
            original_circuit,
            basis_gates=BASIS_GATES,
            seed_transpiler=current_seed,
        )

        target_dir = (
            Path(OUTPUT_DIR) / "disqco" / "mfpqc" / "qv" / f"n{num_qubits}".lower()
        )
        file_base = f"qv_n{num_qubits}_{iteration_id}"
        write_circuit_pair(target_dir, file_base, original_circuit, transpiled_circuit)

## CP Scaling

In [ ]:
CP_SCALING_SIZES = STANDARD_SIZES
CP_SCALING_FRACTIONS = [0.3, 0.5, 0.7, 0.9]

for fraction in tqdm(CP_SCALING_FRACTIONS, desc="CP scaling fractions"):
    fraction_tag = f"f{str(fraction).replace('.', 'p')}"
    for num_qubits in CP_SCALING_SIZES:
        for iteration_id in range(1, NUM_ITERATIONS + 1):
            current_seed = SEED + iteration_id
            random.seed(current_seed)
            np.random.seed(current_seed)

            original_circuit = cp_fraction(
                num_qubits,
                num_qubits,
                fraction=fraction,
                seed=current_seed,
            )
            transpiled_circuit = transpile(
                original_circuit,
                basis_gates=BASIS_GATES,
                seed_transpiler=current_seed,
            )

            target_dir = (
                Path(OUTPUT_DIR)
                / "disqco"
                / "mfpqc"
                / "cp_scaling"
                / fraction_tag.lower()
                / f"n{num_qubits}".lower()
            )
            file_base = f"cp_scaling_{fraction_tag}_n{num_qubits}_{iteration_id}"
            write_circuit_pair(
                target_dir, file_base, original_circuit, transpiled_circuit
            )

## CP Large

In [ ]:
CP_LARGE_SIZES = [112, 128, 144, 160, 176, 192, 208, 224, 240, 256]
CP_LARGE_FRACTION = 0.5

fraction_tag = f"f{str(CP_LARGE_FRACTION).replace('.', 'p')}"
for num_qubits in tqdm(CP_LARGE_SIZES, desc="CP large sizes"):
    for iteration_id in range(1, NUM_ITERATIONS + 1):
        current_seed = SEED + iteration_id
        random.seed(current_seed)
        np.random.seed(current_seed)

        original_circuit = cp_fraction(
            num_qubits,
            num_qubits,
            fraction=CP_LARGE_FRACTION,
            seed=current_seed,
        )
        transpiled_circuit = transpile(
            original_circuit,
            basis_gates=BASIS_GATES,
            seed_transpiler=current_seed,
        )

        target_dir = (
            Path(OUTPUT_DIR)
            / "disqco"
            / "mfpqc"
            / "cp_large"
            / f"n{num_qubits}".lower()
            # / fraction_tag.lower()
        )
        file_base = f"cp_large_n{num_qubits}_{fraction_tag}_{iteration_id}"
        write_circuit_pair(target_dir, file_base, original_circuit, transpiled_circuit)

CP large sizes: 100%|██████████| 1/1 [00:09<00:00,  9.36s/it]


## QASMBench

In [ ]:
QASM_CATEGORIES = ["small", "medium", "large"]
QASM_MAX_DEPTH = 1000
QASM_SKIP_PATTERNS = {"qft", "qv", "QV", "bwt"}

### QASMBench (<50 qubits)

In [ ]:
MIN_QUBITS = 3
MAX_QUBITS = 50

qasm_abs_path = resolve_qasmbench_path(QASMBENCH_PATH)
if not qasm_abs_path.exists():
    raise FileNotFoundError(f"QASMBench directory not found at {qasm_abs_path}")

for category in QASM_CATEGORIES:
    circuits = load_qasmbench_circuits_from_files(
        qasm_abs_path,
        category,
        MIN_QUBITS,
        MAX_QUBITS,
        QASM_MAX_DEPTH,
        BASIS_GATES,
        skip_patterns=QASM_SKIP_PATTERNS,
    )
    for circuit_name, original_circuit, transpiled_circuit in tqdm(
        circuits, desc=f"QASMBench {category} (qasm_50)"
    ):
        target_dir = (
            Path(OUTPUT_DIR)
            / "disqco"
            / "mfpqc"
            / "qasm_50"
            / category.lower()
            / circuit_name.lower()
            / f"n{transpiled_circuit.num_qubits}".lower()
        )
        write_circuit_pair(
            target_dir, circuit_name, original_circuit, transpiled_circuit
        )

## QASMBench (50-100 qubits)

In [ ]:
MIN_QUBITS = 50
MAX_QUBITS = 100

qasm_abs_path = resolve_qasmbench_path(QASMBENCH_PATH)
if not qasm_abs_path.exists():
    raise FileNotFoundError(f"QASMBench directory not found at {qasm_abs_path}")

for category in QASM_CATEGORIES:
    circuits = load_qasmbench_circuits_from_files(
        qasm_abs_path,
        category,
        MIN_QUBITS,
        MAX_QUBITS,
        QASM_MAX_DEPTH,
        BASIS_GATES,
        skip_patterns=QASM_SKIP_PATTERNS,
    )
    for circuit_name, original_circuit, transpiled_circuit in tqdm(
        circuits, desc=f"QASMBench {category} (qasm_100)"
    ):
        target_dir = (
            Path(OUTPUT_DIR)
            / "disqco"
            / "mfpqc"
            / "qasm_100"
            / category.lower()
            / circuit_name.lower()
            / f"n{transpiled_circuit.num_qubits}".lower()
        )
        write_circuit_pair(
            target_dir, circuit_name, original_circuit, transpiled_circuit
        )